# 02 — Iterator Protocol

The previous notebook introduced **iterables** and **iterators**.

This notebook explains the **iterator protocol**: the methods and behavior Python uses to make an object work as an iterator.

The central idea is:

```text
Iterator Protocol
│
├── __iter__()
│
└── __next__()
```

We will learn how these pieces connect to `iter()`, `next()`, and `for` loops. Custom iterator design is covered more deeply in `03_Custom_Iterators.ipynb`.

## 1. Introduction to the Iterator Protocol

In the previous notebook, we saw:

```python
numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))
```

The question now is:

> How does Python know that `iterator` can respond to `iter()` and `next()`?

The answer is the **iterator protocol**.

The iterator protocol defines the behavior Python expects from an iterator.

In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))

## 2. What Is a Protocol in Python?

A **protocol** is a set of methods or behaviors that an object follows so Python knows how to work with it.

For iterators, Python expects two important methods:

```python
__iter__()
__next__()
```

So the central idea is:

```text
Iterator Protocol
│
├── __iter__()
│
└── __next__()
```

We will focus on these methods without going into abstract base classes or advanced protocol design.

## 3. The `__iter__()` Method

`__iter__()` is used to obtain an iterator.

For example, a list can provide an iterator:

In [ ]:
numbers = [10, 20, 30]

iterator = numbers.__iter__()

print(iterator)

This is essentially what:

```python
iter(numbers)
```

does.

In normal Python code, prefer:

```python
iter(numbers)
```

over directly calling:

```python
numbers.__iter__()
```

The dunder-method call is shown here to help you understand the protocol.

In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))

## 4. The `__next__()` Method

The `next()` function asks an iterator for its next value.

Underneath that interface, the iterator provides `__next__()`.

For example:

In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(iterator.__next__())
print(iterator.__next__())

This corresponds conceptually to:

```python
print(next(iterator))
print(next(iterator))
```

The relationship can be pictured as:

```text
next(iterator)
       ↓
iterator.__next__()
       ↓
next value
```

Again, `next(iterator)` is the normal interface to use in everyday Python code.

In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))
print(next(iterator))

## 5. The Two Requirements of an Iterator

An iterator provides two important methods:

```python
class MyIterator:
    def __iter__(self):
        ...

    def __next__(self):
        ...
```

Their roles are:

| Method | Purpose |
|---|---|
| `__iter__()` | Returns an iterator |
| `__next__()` | Returns the next value |
| `StopIteration` | Signals that iteration is finished |

These pieces work together to implement the iterator protocol.

## 6. Understanding `__iter__()` Returning `self`

For an iterator, `__iter__()` normally returns the iterator itself:

In [ ]:
class MyIterator:
    def __iter__(self):
        return self

Why?

Because the object itself is the iterator responsible for producing the next values.

Conceptually:

```text
object
  ↓
__iter__()
  ↓
same object
```

In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(iterator is iter(iterator))

Output:

```text
True
```

An iterator's `__iter__()` returns itself because the iterator is already the object responsible for producing the next values.

This is a useful distinction from a general iterable: an iterable can provide an iterator, while an iterator is already the object performing the iteration.

## 7. Understanding `__next__()`

`__next__()` is responsible for producing the next value.

Conceptually:

```text
__next__()
   ↓
10

__next__()
   ↓
20

__next__()
   ↓
30

__next__()
   ↓
StopIteration
```

The method must eventually signal that there are no more values.

## 8. `StopIteration`

An iterator signals completion by raising:

```python
raise StopIteration
```

Here is a simple iterator that counts from `1` to `3`:

In [ ]:
class Count:
    def __init__(self):
        self.current = 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > 3:
            raise StopIteration

        value = self.current
        self.current += 1
        return value

In [ ]:
counter = Count()

print(next(counter))
print(next(counter))
print(next(counter))

Output:

```text
1
2
3
```

A fourth call has no value left to produce:

In [ ]:
# The next call raises StopIteration.

# print(next(counter))

`StopIteration` is the signal that iteration is finished.

This simple example prepares us for the custom iterator notebook, where we will spend more time designing iterator classes.

## 9. Building a Simple Iterator

Now put the pieces together into a small iterator class.

The iterator will count from `1` up to a supplied limit.

In [ ]:
class CountUp:
    def __init__(self, limit):
        self.current = 1
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current
        self.current += 1
        return value

In [ ]:
counter = CountUp(5)

print(next(counter))
print(next(counter))
print(next(counter))
print(next(counter))
print(next(counter))

Output:

```text
1
2
3
4
5
```

In [ ]:
for number in CountUp(5):
    print(number)

Output:

```text
1
2
3
4
5
```

This is only a first, simple custom iterator example. `03_Custom_Iterators.ipynb` will focus much more deeply on designing custom iterator classes.

## 10. How `for` Loops Use the Iterator Protocol

Consider this familiar code:

In [ ]:
numbers = [10, 20, 30]

for number in numbers:
    print(number)

Conceptually, Python performs an operation equivalent to:

```python
iterator = iter(numbers)

while True:
    try:
        number = next(iterator)
        print(number)
    except StopIteration:
        break
```

The protocol connection is:

```text
for
 ↓
iter()
 ↓
__iter__()
 ↓
iterator
 ↓
next()
 ↓
__next__()
 ↓
value
 ↓
repeat
 ↓
StopIteration
 ↓
loop ends
```

In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

while True:
    try:
        number = next(iterator)
        print(number)
    except StopIteration:
        break

This is a **conceptual/manual recreation** of what the `for` loop accomplishes. The actual interpreter implementation has additional details.

The key mental model is:

```text
iterable
    ↓
iter()
    ↓
iterator
    ↓
next()
    ↓
__next__()
    ↓
value
    ↓
repeat
    ↓
StopIteration
```

## 11. Manually Using the Iterator Protocol

We can explicitly use the normal iterator interfaces:

In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))
print(next(iterator))
print(next(iterator))

We can also directly call the underlying dunder methods:

In [ ]:
numbers = [10, 20, 30]

iterator = numbers.__iter__()

print(iterator.__next__())
print(iterator.__next__())
print(iterator.__next__())

Both examples produce:

```text
10
20
30
```

In normal Python code, prefer:

```python
iter(numbers)
next(iterator)
```

The dunder methods are being shown here so you can understand how the iterator protocol works.

## 12. Iterable vs Iterator Protocol

Now consolidate the concepts from the previous two notebooks.

### Iterable

An iterable can provide an iterator:

```python
numbers = [10, 20, 30]

iterator = iter(numbers)
```

### Iterator

An iterator provides the next value:

```python
next(iterator)
```

Conceptually:

```text
Iterable
   │
   │ iter()
   ↓
Iterator
   │
   │ next()
   ↓
Value
```

The iterator protocol is based on:

```text
__iter__()
__next__()
StopIteration
```

A useful distinction is:

> An iterable can give you an iterator. An iterator is the object that produces the values and remembers its iteration state.

## 13. Common Mistakes

### Mistake 1 — Missing `__iter__()`

This class only defines `__next__()`:

In [ ]:
class MyIterator:
    def __next__(self):
        return 1

It does not provide the complete iterator protocol expected by normal iterator usage.

A complete iterator needs both:

```python
__iter__()
__next__()
```

### Mistake 2 — Missing `__next__()`

This class only defines `__iter__()`:

In [ ]:
class MyIterator:
    def __iter__(self):
        return self

Again, it is incomplete because there is no method that produces the next value.

The two methods have different jobs:

```text
__iter__() → provide the iterator
__next__() → provide the next value
```

### Mistake 3 — Forgetting `StopIteration`

Suppose an iterator is supposed to end, but `__next__()` never signals completion:

In [ ]:
class BadCounter:
    def __init__(self):
        self.current = 1

    def __iter__(self):
        return self

    def __next__(self):
        value = self.current
        self.current += 1
        return value

This iterator has no stopping condition. A `for` loop would keep requesting values indefinitely.

If the iterator is supposed to finish, `__next__()` must eventually raise:

```python
raise StopIteration
```

### Mistake 4 — Returning a New Unrelated Object from an Iterator's `__iter__()`

For an iterator, the normal pattern is:

```python
def __iter__(self):
    return self
```

The iterator returns itself because it is already the object responsible for producing the next values.

We will explore this behavior properly when building custom iterators.

## 14. Summary

The iterator protocol can be pictured as:

```text
Iterator Protocol
│
├── __iter__()
│      ↓
│   returns iterator
│
├── __next__()
│      ↓
│   returns next value
│
└── StopIteration
       ↓
    iteration ends
```

### Key Points

- Python uses protocols to define expected object behavior.
- The iterator protocol is based on `__iter__()` and `__next__()`.
- `iter(obj)` obtains an iterator.
- `next(iterator)` requests the next value.
- An iterator's `__iter__()` normally returns `self`.
- `__next__()` produces one value at a time.
- `StopIteration` signals that no values remain.
- `for` loops rely on this protocol.

### Coming Next

Keep the deeper topics in their dedicated notebooks:

```text
Custom iterator classes      → 03_Custom_Iterators
Generators                   → 04_Generators
yield                        → 05_Yield_and_Generator_Functions
Generator expressions        → 06_Generator_Expressions
Performance/memory analysis  → 07_Iterators_vs_Generators
```

The progression is:

```text
01_Iterables_and_Iterators
        ↓
02_Iterator_Protocol
        ↓
03_Custom_Iterators
        ↓
04_Generators
```

The learner first understands **what iterators are**, then **the protocol that makes them work**, and only then starts **building their own**.